# Prompt Engineering Basics — AI Engineering Interview Prep

Zero-shot, few-shot, and role prompting with Claude API.

**Prerequisites**: Set `ANTHROPIC_API_KEY` environment variable.

In [ ]:
import anthropic
import os

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODEL = "claude-haiku-4-5-20251001"

def ask(prompt, system=None, max_tokens=300):
    kwargs = {"model": MODEL, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    resp = client.messages.create(**kwargs)
    return resp.content[0].text

print(f"Model: {MODEL}")

## 1. Zero-Shot Prompting

In [ ]:
# Zero-shot: no examples, just a direct instruction
zero_shot_prompt = "Classify the sentiment of this review as Positive, Negative, or Neutral:\n\nReview: 'The model training was faster than expected but the accuracy was disappointing.'"

result = ask(zero_shot_prompt)
print("Zero-shot sentiment classification:")
print(result)

# Zero-shot with explicit format instruction
formatted_prompt = """Classify the sentiment of each review. Respond with ONLY the label: Positive, Negative, or Neutral.

Review: 'The model training was faster than expected but the accuracy was disappointing.'"""

result2 = ask(formatted_prompt)
print("\nWith format instruction:")
print(result2)

## 2. Few-Shot Prompting

In [ ]:
# Few-shot: provide labeled examples before the target
few_shot_prompt = """Classify ML topics into: Supervised, Unsupervised, or Reinforcement Learning.

Topic: Predicting house prices from features → Supervised
Topic: Grouping customers by purchase behavior → Unsupervised
Topic: Training a robot to walk → Reinforcement Learning
Topic: Email spam detection → Supervised
Topic: Finding anomalies in server logs → Unsupervised

Topic: Teaching an agent to play chess →"""

result = ask(few_shot_prompt)
print("Few-shot ML topic classification:")
print(result)

# Few-shot for structured extraction
extraction_prompt = """Extract model metrics from text. Output: Model: <name> | Accuracy: <value> | Dataset: <name>

Text: 'We trained a Random Forest and achieved 94.2% accuracy on MNIST.' → Model: Random Forest | Accuracy: 94.2% | Dataset: MNIST
Text: 'The BERT model reached 91% F1 on SQuAD 2.0.' → Model: BERT | Accuracy: 91% F1 | Dataset: SQuAD 2.0

Text: 'Our XGBoost classifier hit 88.7% accuracy on the ImageNet validation set.' →"""

result2 = ask(extraction_prompt)
print("\nFew-shot structured extraction:")
print(result2)

## 3. Role Prompting

In [ ]:
# Role prompting via system prompt
roles = [
    {
        "name": "ML Interviewer",
        "system": "You are a senior ML engineer at a FAANG company conducting a technical interview. Ask probing follow-up questions and be concise. Max 60 words.",
        "user": "Can you explain how gradient descent works?"
    },
    {
        "name": "Tutor",
        "system": "You are a patient ML tutor for beginners. Use simple analogies and avoid jargon. Max 80 words.",
        "user": "Can you explain how gradient descent works?"
    },
    {
        "name": "Code Reviewer",
        "system": "You are a strict code reviewer. Point out only critical bugs and performance issues. Be terse. Max 50 words.",
        "user": "Can you explain how gradient descent works?"
    }
]

for role in roles:
    result = ask(role["user"], system=role["system"])
    print(f"[{role['name']}]")
    print(result)
    print("-" * 50)

## 4. Prompt Templates

In [ ]:
# Reusable prompt templates
def explain_concept(concept, audience="ML engineer", max_words=100):
    return f"""Explain {concept} to a {audience}. Keep it under {max_words} words.
Structure: 1 sentence definition → 1 concrete example → 1 key tradeoff."""

def compare_concepts(a, b):
    return f"""Compare {a} vs {b} for ML. Use this exact format:
| Aspect | {a} | {b} |
|--------|------|------|
Fill in 4 rows: Use case, Speed, Memory, Best for."""

# Test templates
concepts = ["dropout", "batch normalization"]
for c in concepts:
    print(f"--- {c} ---")
    print(ask(explain_concept(c)))
    print()

print("--- Comparison ---")
print(ask(compare_concepts("L1 regularization", "L2 regularization")))

## 5. Output Format Control

In [ ]:
import json

# Force JSON output
json_prompt = """Given this ML experiment result, output ONLY a JSON object with keys:
model_name, accuracy (float), training_time_seconds (int), hyperparameters (dict), recommendation (string).

Experiment: We trained ResNet-50 with lr=0.001, batch_size=32, epochs=50 on CIFAR-10.
Final test accuracy: 93.4%. Training took about 2 hours on a single GPU."""

raw = ask(json_prompt, max_tokens=400)
print("Raw response:")
print(raw)

# Parse JSON (strip markdown code fences if present)
clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
try:
    parsed = json.loads(clean)
    print("\nParsed JSON:")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"Parse error: {e}")

## Interview Tips

- **Zero-shot**: best starting point; add examples only when zero-shot fails.
- **Few-shot order**: put the most representative example last (recency bias).
- **Role clarity**: specific roles ("senior ML engineer at FAANG") beat generic ones ("expert").
- **Format instructions**: say EXACTLY what output format you want. Use delimiters like `→` to separate input from expected output.
- **Prompt length**: shorter is often better — every token in the prompt is a token out of the context window.
- **Temperature 0**: for deterministic, structured output. Higher for creative tasks.